In [1]:
import os
from dotenv import load_dotenv
from dbconnect import *

import asyncio

import polars as pl
import pandas as pd
import numpy as np

from decimal import Decimal
from datetime import datetime

load_dotenv()

url = os.getenv("DATABASE_URL")
connect_db = DBConnect(url)

In [2]:
async def get_data(days_filter: tuple) -> dict:

    if isinstance(days_filter, int):
        days_filter = (days_filter,)

    async def fetch_table(key, query, params=None):
        try:
            data = await asyncio.to_thread(connect_db.execute_query, query, params)
            
            return key, data if isinstance(data, list) else []
        except Exception as e:
            print(f"Error fetching {key}: {e}")
            return key, []
    
    # view table
    # sql_view_template = "select * from digipro.{} where extract(day from date) in :days"
    
    # tables = {
    #     "scheduled_downtime": "scheduled_downtime",
    #     "not_scheduled_downtime": "not_scheduled_downtime",
    #     "kode_pakan_gabungan": "kode_pakan_gabungan",
    #     "operating_time_per_kode_pakan": "operating_time_per_kode_pakan",
    # }

    tasks = []

    # for k, v in tables.items():
    #     tasks.append(fetch_table(k, sql_view_template.format(v), {"days": days_filter}))

    # scheduled downtime
    sql_sd = """
        select
            t1.line,
            t1.shift,
            t1.machine_losses_lvl_1,
            t1.machine_losses_lvl_2,
            t1.machine_losses_lvl_3,
            t1.scheduled_downtime
        from digipro.scheduled_downtime t1
        where extract(day from t1.date) in :days
        and t1.plant = 'CKP'
        and t1.business_unit = 'fish'
    """
    tasks.append(fetch_table("scheduled_downtime", sql_sd, {"days": days_filter}))

    # not scheduled downtime
    sql_nsd = """
        select
            t1.line,
            t1.shift,
            t1.machine_losses_lvl_1,
            t1.machine_losses_lvl_2,
            t1.machine_losses_lvl_3,
            t1.not_scheduled_downtime
        from digipro.not_scheduled_downtime t1
        where extract(day from t1.date) in :days
        and t1.plant = 'CKP'
        and t1.business_unit = 'fish'
    """
    tasks.append(fetch_table("not_scheduled_downtime", sql_nsd, {"days": days_filter}))

    # kode pakan gabungan
    sql_kpg = """
        select
            t1.line,
            t1.shift,
            t1.kode_pakan_gabungan
        from digipro.kode_pakan_gabungan t1
        where extract(day from t1.date) in :days
        and t1.plant = 'CKP'
        and t1.business_unit = 'fish'
    """
    tasks.append(fetch_table("kode_pakan_gabungan", sql_kpg, {"days": days_filter}))

    # standard throughput
    sql_std_throughput = """
        select
            t1.line,
            t1.shift,
            t1.final_operating_time,
            t1.std_output
        from digipro.standard_throughput t1
        where extract(day from t1.date) in :days
        and t1.plant = 'CKP'
        and t1.business_unit = 'fish'
    """
    tasks.append(fetch_table("standard_throughput", sql_std_throughput, {"days": days_filter}))

    # output
    sql_output = """
        select
            t1.line,
            t1.shift,
            t1.kode_pakan,
            t1.type,
            t1.weight
        from digipro.output t1
        where t1.is_active = true
        and extract(day from t1.date) in :days
        and t1.plant = 'CKP'
        and t1.business_unit = 'fish'
    """
    tasks.append(fetch_table("output", sql_output, {"days": days_filter}))

    # master line
    sql_master_line = """
        select 
            t1.line 
        from digipro.master_line t1 
        where t1.is_active = true
    """
    tasks.append(fetch_table("master_line", sql_master_line))

    # merged line
    sql_merged_line = """
        select
        	t1.transaction_id,
        	array_agg(t3.line) as merged_line
        from digipro.transactions_line t1
        left join digipro.transactions_line_details t2 
        	on t1.transaction_id = t2.transaction_id
        left join digipro.master_line t3 
        	on t2.line_id = t3.id
        where
        	t1.is_active = true
        group by 
        	t1.transaction_id
    """
    tasks.append(fetch_table("merged_line", sql_merged_line))

    # master shift
    sql_master_shift = """
        select distinct
	        t1.shift
        from digipro.master_shift t1
        where t1.plant = 'CKP'
        and t1.business_unit = 'fish'
        order by t1.shift
    """

    tasks.append(fetch_table("master_shift", sql_master_shift))

    sql_losses_sd = """
        select distinct
	        t1.id,
	        t2.name as machine_losses_lvl_1,
	        t3.name as machine_losses_lvl_2,
	        t4.name as machine_losses_lvl_3
        from digipro.master_machine_losses t1
        left join digipro.master_machine_losses_lvl_1 t2 on t1.ml_lvl_1_id = t2.id
        left join digipro.master_machine_losses_lvl_2 t3 on t1.ml_lvl_2_id = t3.id
        left join digipro.master_machine_losses_lvl_3 t4 on t1.ml_lvl_3_id = t4.id
        where t2.name = 'Scheduled Downtime'
        order by t1.id
    """

    tasks.append(fetch_table("master_losses_sd", sql_losses_sd))

    sql_losses_nsd = """
        select distinct
	        t1.id,
	        t2.name as machine_losses_lvl_1,
	        t3.name as machine_losses_lvl_2,
	        t4.name as machine_losses_lvl_3
        from digipro.master_machine_losses t1
        left join digipro.master_machine_losses_lvl_1 t2 on t1.ml_lvl_1_id = t2.id
        left join digipro.master_machine_losses_lvl_2 t3 on t1.ml_lvl_2_id = t3.id
        left join digipro.master_machine_losses_lvl_3 t4 on t1.ml_lvl_3_id = t4.id
        where t2.name not in ('Scheduled Downtime', 'Performance Loss', 'Defects & Rework Loss')
        order by t1.id
    """

    tasks.append(fetch_table("master_losses_nsd", sql_losses_nsd))    

    try:
        results = dict(await asyncio.gather(*tasks))
        return results
    except Exception as e:
        print(f"Error gathering tasks: {e}")
        return {}

In [31]:
def total_time(data: dict, days_filter: tuple, is_total: bool = False, is_json: bool = False):

    if not data or not days_filter:
        return [] if is_json else pl.DataFrame
    
    if isinstance(days_filter, int):
        days_filter = (days_filter,)
    
    if len(days_filter) == 2:
        start, end = days_filter
        total_days = (end - start) + 1
    else:
        total_days = 1
        
    total_time = float(total_days * 8)
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    df_shifts = pl.from_dicts(shift_data)

    df_base = df_master_line.join(df_shifts, how='cross').with_columns(
        pl.lit(total_time).alias("total_time")
    )

    additional_dfs = []
    for item in merged_line_data:
        target_lines = item['merged_line']
        joined_name = " & ".join(target_lines)
        
        df_temp = (
            df_base.filter(pl.col('line').is_in(target_lines))
            .group_by(['shift'])
            .agg(pl.col('total_time').sum())
            .with_columns(pl.lit(joined_name).alias('line'))
        )
        additional_dfs.append(df_temp)

    if additional_dfs:
        df_merged_all = pl.concat(additional_dfs)
        df = pl.concat([df_base, df_merged_all], how="diagonal")
    else:
        df = df_base

    if is_total:
        df = (
            df.group_by(['line'])
            .agg(pl.col('total_time').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])

    return df.to_dicts() if is_json else df

def scheduled_downtime(data: dict, is_total: bool = False, is_json: bool = False):

    if not data['scheduled_downtime']:
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']
    master_losses_data = data['master_losses_sd']
    sd_data = data['scheduled_downtime']

    df_shifts = pl.from_dicts(shift_data)
    df_master_losses = pl.from_dicts(master_losses_data)

    df_raw = pl.from_dicts(sd_data)
    df_raw = df_raw.with_columns([
        pl.col(col).cast(pl.Float64) 
        for col, dtype in df_raw.schema.items() 
        if isinstance(dtype, pl.Decimal)
    ])

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    skeleton = (
        df_all_lines 
        .join(df_shifts, how="cross")
        .join(df_master_losses, how="cross")
    )

    df_base_grouped = df_raw.group_by([
        'shift', 'line', 'machine_losses_lvl_1', 
        'machine_losses_lvl_2', 'machine_losses_lvl_3'
    ]).agg(pl.col('scheduled_downtime').sum().alias('total_scheduled_downtime'))

    additional_dfs = []
    for merged_line in merged_line_data:
        target_lines = merged_line['merged_line']
        joined_line_names = " & ".join(target_lines)

        temp_merged = (
            df_base_grouped.filter(pl.col('line').is_in(target_lines))
            .group_by([
                'shift', 'machine_losses_lvl_1', 
                'machine_losses_lvl_2', 'machine_losses_lvl_3'
            ])
            .agg(pl.col('total_scheduled_downtime').sum())
            .with_columns(pl.lit(joined_line_names).alias('line'))
            .select(df_base_grouped.columns)
        )
        additional_dfs.append(temp_merged)

    if additional_dfs:
        df_all_data = pl.concat([df_base_grouped] + additional_dfs)
    else:
        df_all_data = df_base_grouped

    df = (
        skeleton.join(
            df_all_data,
            on=['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'],
            how='left'
        )
        .with_columns(pl.col('total_scheduled_downtime').fill_null(0.0))
    ).select([ 
            'line', 
            'shift', 
            'machine_losses_lvl_1', 
            'machine_losses_lvl_2', 
            'machine_losses_lvl_3', 
            'total_scheduled_downtime'
        ])
    
    if is_total:
        df = (
            df.group_by([
                'line',
                'machine_losses_lvl_1', 
                'machine_losses_lvl_2', 
                'machine_losses_lvl_3', 
            ])
            .agg(pl.col('total_scheduled_downtime').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])
    
    return df.to_dicts() if is_json else df


def loading_time(total_time_data: pl.DataFrame, scheduled_downtime_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):

    sd_summary = (
        scheduled_downtime_data.group_by(['shift', 'line'] if not is_total else ['line'])
        .agg(pl.col('total_scheduled_downtime').sum())
    )

    df = (
        total_time_data.join(sd_summary, on=['shift', 'line'] if not is_total else ['line'], how="left")
        .with_columns(pl.col('total_scheduled_downtime').fill_null(0))
        .with_columns(
            (pl.col('total_time') - pl.col('total_scheduled_downtime')).alias('loading_time')
        )
    ).select(['line', 'shift', 'loading_time'] if not is_total else ['line', 'loading_time'])   

    if is_total:
        df = (
            df.group_by(['line'])
            .agg(pl.col('loading_time').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])

    return df.to_dicts() if is_json else df


def not_scheduled_downtime(data: dict, is_total: bool = False, is_json: bool = False):

    if not data['not_scheduled_downtime']:
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']
    master_losses_data = data['master_losses_nsd']
    nsd_data = data['not_scheduled_downtime']

    df_shifts = pl.from_dicts(shift_data)
    df_master_losses = pl.from_dicts(master_losses_data)
    
    df_raw = pl.from_dicts(nsd_data)
    df_raw = df_raw.with_columns([
        pl.col(col).cast(pl.Float64) 
        for col, dtype in df_raw.schema.items() 
        if isinstance(dtype, pl.Decimal)
    ])    

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])  

    skeleton = (
        df_all_lines 
        .join(df_shifts, how="cross")
        .join(df_master_losses, how="cross")
    )

    df_base_grouped = df_raw.group_by([
        'shift', 'line', 'machine_losses_lvl_1', 
        'machine_losses_lvl_2', 'machine_losses_lvl_3'
    ]).agg(pl.col('not_scheduled_downtime').sum().alias('total_not_scheduled_downtime'))      
    
    additional_dfs = []
    for merged_line in merged_line_data:
        target_lines = merged_line['merged_line']
        joined_line_names = " & ".join(target_lines)

        temp_merged = (
            df_base_grouped.filter(pl.col('line').is_in(target_lines))
            .group_by([
                'shift', 'machine_losses_lvl_1', 
                'machine_losses_lvl_2', 'machine_losses_lvl_3'
            ])
            .agg(pl.col('total_not_scheduled_downtime').sum())
            .with_columns(pl.lit(joined_line_names).alias('line'))
            .select(df_base_grouped.columns)
        )
        additional_dfs.append(temp_merged)

    if additional_dfs:
        df_all_data = pl.concat([df_base_grouped] + additional_dfs)
    else:
        df_all_data = df_base_grouped

    df = (
        skeleton.join(
            df_all_data,
            on=['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'],
            how='left'
        )
        .with_columns(pl.col('total_not_scheduled_downtime').fill_null(0.0))
    ).select([
            'line', 
            'shift', 
            'machine_losses_lvl_1', 
            'machine_losses_lvl_2', 
            'machine_losses_lvl_3', 
            'total_not_scheduled_downtime'
        ])
    
    if is_total:
        df = (
            df.group_by([
                'line',
                'machine_losses_lvl_1', 
                'machine_losses_lvl_2', 
                'machine_losses_lvl_3', 
            ])
            .agg(pl.col('total_not_scheduled_downtime').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])
    
    return df.to_dicts() if is_json else df

def operating_time(loading_time_data: pl.DataFrame, not_scheduled_downtime_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    nsd_summary = (
        not_scheduled_downtime_data.group_by(['shift', 'line'] if not is_total else ['line'])
        .agg(pl.col('total_not_scheduled_downtime').sum())
    )

    df = (
        loading_time_data.join(nsd_summary, on=['shift', 'line'] if not is_total else ['line'], how="left")
        .with_columns(pl.col('total_not_scheduled_downtime').fill_null(0))
        .with_columns(
            (pl.col('loading_time') - pl.col('total_not_scheduled_downtime')).alias('operating_time')
        )
    ).select(['line', 'shift', 'operating_time'] if not is_total else ['line', 'operating_time'])    

    if is_total:
        df = (
            df.group_by(['line'])
            .agg(pl.col('operating_time').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])
    
    return df.to_dicts() if is_json else df

def performance_loss(data: dict, operating_time_data: pl.DataFrame, output_data: pl.DataFrame, standard_throughput_data: pl.DataFrame, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift'
    ]
    
    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    df_shifts = pl.from_dicts(shift_data)

    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_speed_loss = df_all_lines.join(df_shifts, how="cross")

    df_speed_loss = df_speed_loss.with_columns(
        pl.lit("Performance Loss").alias("machine_losses_lvl_1"),
        pl.lit("Speed Loss").alias("machine_losses_lvl_2"),
        pl.lit("Speed Loss").alias("machine_losses_lvl_3")
    )

    df_total_output = output_data.group_by(
        ['shift', 'line']
        ).agg(pl.col('total_output').sum().alias('total_output'))

    df_total_output = df_total_output.sort(['shift', 'line'])

    df_std_throughput = standard_throughput_data.group_by(
        ['shift', 'line']
        ).agg(pl.col('std_throughput').sum())

    df_join = (
        df_speed_loss.join(df_total_output, on=['shift', 'line'], how='left')
        .with_columns(pl.col('total_output').fill_null(0))
    )

    df_join_2 = (
        df_join.join(df_std_throughput, on=['shift', 'line'], how='left')
        .with_columns(pl.col('std_throughput').fill_null(0))
        .with_columns(
            pl.when(pl.col('std_throughput') > 0)
            .then(pl.col('total_output') / pl.col('std_throughput'))
            .otherwise(0)
            .round(2)
            .alias('temp_performance_loss')
        )
    )

    df = (
        df_join_2.join(operating_time_data, on=['shift', 'line'], how='left')
        .with_columns(
            (pl.col('operating_time') - pl.col('temp_performance_loss'))
            .round(2)
            .alias('performance_loss')
        )
    ).select([
        'line', 
        'shift', 
        'machine_losses_lvl_1',
        'machine_losses_lvl_2',
        'machine_losses_lvl_3',
        'performance_loss'
        ])
    
    df = df.sort(['line', 'shift'])

    return df.to_dicts() if is_json else df

def performance_loss_total(performance_loss_data: pl.DataFrame, is_json: bool = False):
    if performance_loss_data is None or performance_loss_data.is_empty():
        return []
    
    df = (
        performance_loss_data.group_by([
            'line',
            'machine_losses_lvl_1',
            'machine_losses_lvl_2',
            'machine_losses_lvl_3',
        ]).agg(pl.col('performance_loss').sum().alias('total_performance_loss'))
    ).select([
        'line',
        'machine_losses_lvl_1',
        'machine_losses_lvl_2',
        'machine_losses_lvl_3',        
        'total_performance_loss'
    ]).sort(['line'])

    return df.to_dicts() if is_json else df  
    
def net_operating_time(operating_time_data: pl.DataFrame, spd_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    df = (
        operating_time_data.join(spd_data, on=['shift', 'line'] if not is_total else ['line'], how="left")
        .with_columns(pl.col('performance_loss' if not is_total else 'total_performance_loss').fill_null(0))
        .with_columns(
            (pl.col('operating_time') - pl.col('performance_loss' if not is_total else 'total_performance_loss')).alias('net_operating_time')
        )
    ).select(['line', 'shift','net_operating_time'] if not is_total else ['line', 'net_operating_time']) 

    if is_total:
        df = (
            df.group_by(['line'])
            .agg(pl.col('net_operating_time').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])
    
    return df.to_dicts() if is_json else df

def defect_rework_loss(data: dict, output_data: pl.DataFrame, actual_throughput_data: pl.DataFrame, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift'
    ]
    
    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    df_shifts = pl.from_dicts(shift_data)

    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_defect_rework_loss = df_all_lines.join(df_shifts, how="cross")

    df_defect_rework_loss = df_defect_rework_loss.with_columns(
        pl.lit("Defects & Rework Loss").alias("machine_losses_lvl_1"),
        pl.lit("Defects & Rework Loss").alias("machine_losses_lvl_2"),
        pl.lit("Defects & Rework Loss").alias("machine_losses_lvl_3")
    )
    
    df_total_output = output_data.group_by(
        ['shift', 'line']
    ).agg(pl.col('total_output').sum().alias('total_output'))

    df_total_output = df_total_output.sort(['shift', 'line'])

    df_defect_rework_output = output_data.filter(
        pl.col('type').is_in(['WIP (kg)', 'Remix (kg)', 'Reject (kg)'])
    ).group_by(
        ['shift', 'line']
    ).agg(pl.col('total_output').sum().alias('total_output_defect_rework'))

    df_join = (
        df_defect_rework_loss.join(df_defect_rework_output, on=['shift', 'line'], how='left')
        .with_columns(pl.col('total_output_defect_rework').fill_null(0))
    )

    df = (
        df_join.join(actual_throughput_data, on=['shift', 'line'], how='left')
        .with_columns(
            pl.when(pl.col('actual_throughput') > 0)
            .then(pl.col('total_output_defect_rework') / pl.col('actual_throughput'))
            .otherwise(0)
            .round(2)
            .alias('defect_rework_loss')
        )
    ).select([
        'line',
        'shift',
        'machine_losses_lvl_1',
        'machine_losses_lvl_2',
        'machine_losses_lvl_3',
        'defect_rework_loss'
    ])

    df = df.sort(['line', 'shift'])

    return df.to_dicts() if is_json else df

def defect_rework_loss_total(defect_rework_loss_data: pl.DataFrame, is_json: bool = False):
    if defect_rework_loss_data is None or defect_rework_loss_data.is_empty():
        return []
    
    df = (
        defect_rework_loss_data.group_by([
            'line',
            'machine_losses_lvl_1',
            'machine_losses_lvl_2',
            'machine_losses_lvl_3',
        ]).agg(pl.col('defect_rework_loss').sum().alias('total_defect_rework_loss'))
    ).select([
        'line',
        'machine_losses_lvl_1',
        'machine_losses_lvl_2',
        'machine_losses_lvl_3',
        'total_defect_rework_loss'
    ]).sort(['line'])

    return df.to_dicts() if is_json else df  

def value_operating_time(net_operating_time_data: pl.DataFrame, defect_rework_loss_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    df = (
        net_operating_time_data.join(defect_rework_loss_data, on=['shift', 'line'] if not is_total else ['line'], how='left')
        .with_columns(
            (pl.col('net_operating_time') - pl.col('defect_rework_loss' if not is_total else 'total_defect_rework_loss')).alias('value_net_operating_time')
        )
    ).select(['line', 'shift','value_net_operating_time'] if not is_total else ['line', 'value_net_operating_time']) 

    if is_total:
        df = (
            df.group_by(['line'])
            .agg(pl.col('value_net_operating_time').sum())
            .sort(['line'])
        )
    else:
        df = df.sort(['line', 'shift'])    

    return df.to_dicts() if is_json else df

def standard_throughput(data: dict, is_total: bool = False, is_json: bool = False):
    if not data['standard_throughput']:
        return [] if is_json else pl.DataFrame()

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']
    std_throughput_data = data['standard_throughput']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df = df_all_lines
    if not is_total:
        df_shifts = pl.from_dicts(shift_data)
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    df_std_throughput = pl.from_dicts(std_throughput_data)
    df_std_throughput = df_std_throughput.with_columns([
        pl.col(col).cast(pl.Float64) 
        for col, dtype in df_std_throughput.schema.items() 
        if isinstance(dtype, pl.Decimal)
    ])

    df_std_throughput_shift = df_std_throughput.group_by(
           ['line', 'shift']
        ).agg([
            pl.col('final_operating_time').sum(),
            pl.col('std_output').sum()
        ]).with_columns(
            pl.when(pl.col('final_operating_time') > 0)
            .then(pl.col('std_output') / pl.col('final_operating_time'))
            .otherwise(0)
            .alias('std_throughput')
        ).select(['line', 'shift', 'std_throughput'])

    df_std_throughput_line = df_std_throughput.group_by(
           ['line']
        ).agg([
            pl.col('final_operating_time').sum(),
            pl.col('std_output').sum()
        ]).with_columns(
            pl.when(pl.col('final_operating_time') > 0)
            .then(pl.col('std_output') / pl.col('final_operating_time'))
            .otherwise(0)
            .alias('std_throughput')
        ).select(['line', 'std_throughput'])

    additional_dfs = []

    for merged_line in merged_line_data:
        target_lines = merged_line['merged_line']
        joined_line_names = " & ".join(target_lines)

        if not is_total:
            temp_df = (
                df_std_throughput_shift.filter(pl.col('line').is_in(target_lines))
                .group_by(['shift'])
                .agg(pl.col('std_throughput').sum())
                .with_columns(pl.lit(joined_line_names).alias('line'))
            )
        else:
            temp_df = (
                df_std_throughput_shift.filter(pl.col('line').is_in(target_lines))
                .group_by(['shift'])
                .agg(pl.col('std_throughput').sum())
                .select(
                    pl.col('std_throughput')
                    .filter(pl.col('std_throughput') > 0)
                    .mean()
                    .fill_null(0)
                    .alias('std_throughput')
                )
                .with_columns(pl.lit(joined_line_names).alias('line'))
            )

        temp_df = temp_df.select(['line', 'shift', 'std_throughput'] if not is_total else ['line', 'std_throughput'])
        additional_dfs.append(temp_df)

    if additional_dfs:
        df_std_throughput = pl.concat([df_std_throughput_shift] if not is_total else [df_std_throughput_line] + additional_dfs)

    df = (
        df.join(df_std_throughput_shift if not is_total else df_std_throughput_line, on=['shift', 'line'] if not is_total else ['line'], how='left')
        .with_columns(pl.col('std_throughput').fill_null(0))
    )   

    df = df.sort(['line'])

    return df.to_dicts() if is_json else df 

def actual_throughput(operating_time_data: pl.DataFrame, output_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    df_total_output = output_data.group_by(
        ['shift', 'line'] if not is_total else ['line']
    ).agg(pl.col('total_output').sum().alias('total_output'))

    df_actual_throughput = (
        df_total_output.join(operating_time_data, on=['shift', 'line'] if not is_total else ['line'], how='left')
        .with_columns(
            pl.when(pl.col('operating_time') > 0)
            .then(pl.col('total_output') / pl.col('operating_time'))
            .otherwise(0)
            .round(2)
            .alias('actual_throughput')
        )
    ).select(['line', 'shift','actual_throughput'] if not is_total else ['line', 'actual_throughput']) 

    if is_total:
        df = df_actual_throughput.sort(['line'])
    else:
        df = df_actual_throughput.sort(['line', 'shift'])

    return df.to_dicts() if is_json else df      

def output(data: dict, is_total: bool = False, is_json: bool = False):
    if not data['output']:
        return [] if is_json else pl.DataFrame()
    
    output_data = data['output']
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df_output = pl.from_dicts(output_data)
    df_output = df_output.with_columns([
        pl.col(col).cast(pl.Float64) 
        for col, dtype in df_output.schema.items() 
        if isinstance(dtype, pl.Decimal)
    ])

    df_output = df_output.group_by(
        ['shift', 'line', 'type']
        ).agg(pl.col('weight').sum().alias('total_output'))
    
    unique_types = df_output.select(pl.col('type').unique()) # ini hati-hati, soalnya hanya berdasarkan data yang ada, bukan master

    df = (
        df_all_lines
        .join(df_shifts, how="cross")
        .join(unique_types, how="cross")
    )
    
    additional_dfs = []

    for merged_line in merged_line_data:
        target_lines = merged_line['merged_line']
        joined_line_names = " & ".join(target_lines)

        temp_df = (
            df_output.filter(pl.col('line').is_in(target_lines))
            .group_by([
                'shift', 'type'
            ])
            .agg(pl.col('total_output').sum())
            .with_columns(pl.lit(joined_line_names).alias('line'))
            .select(df_output.columns)
        )

        additional_dfs.append(temp_df)
    
    if additional_dfs:
        df_output = pl.concat([df_output] + additional_dfs)

    df = (
        df.join(df_output, on=['shift', 'line', 'type'], how='left')
        .with_columns(pl.col('total_output').fill_null(0))
    )

    if is_total:
        df = (
            df.group_by(['line', 'type'])
            .agg(pl.col('total_output').sum())
            .sort(['line', 'type'])
        )
    else:
        df = df.sort(['shift', 'line', 'type'])

    return df.to_dicts() if is_json else df

def kode_pakan_gabungan(data: dict, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
        'kode_pakan_gabungan'
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']
    kode_pakan_gabungan_data = data['kode_pakan_gabungan']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)
    df_kpg = pl.from_dicts(kode_pakan_gabungan_data)

    df_kpg_total = df_kpg.group_by(
        ['line', 'shift']
    ).agg(
        pl.col('kode_pakan_gabungan').sort().str.join(';')
    )

    df = (
        df_all_lines
        .join(df_shifts, how="cross")
    )

    df = (
        df.join(df_kpg_total, on=['shift', 'line'], how='left')
        .with_columns(pl.col('kode_pakan_gabungan').fill_null(''))
    ).sort(['line', 'shift'])

    return df.to_dicts() if is_json else df

def availability_rate(data: dict, operating_time_data: pl.DataFrame, loading_time_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift'
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    df_join = (
        df.join(operating_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left')
        .with_columns(pl.col('operating_time').fill_null(0))
    ).join(
        loading_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
    ).with_columns(pl.col('loading_time').fill_null(0))

    df = df_join.with_columns(
        pl.when(
            (pl.col('loading_time') > 0)
        )
        .then((pl.col('operating_time') / pl.col('loading_time')) * 100)
        .otherwise(None)
        .round(2)
        .alias('availability_rate')
    ).select(
        ['line', 'shift', 'availability_rate'] if not is_total else ['line', 'availability_rate']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    return df.to_dicts() if is_json else df

def performance_rate(data: dict, actual_throughput_data: pl.DataFrame, standard_throughput_data: pl.DataFrame, is_total: bool = False, is_fixed: bool = False, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    df_join = (
        df.join(actual_throughput_data, on=['line', 'shift'] if not is_total else ['line'], how='left')
        .with_columns(pl.col('actual_throughput').fill_null(0))
    ).join(
        standard_throughput_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
    ).with_columns(pl.col('std_throughput').fill_null(0))

    individual_lines_in_merged = []
    for item in merged_line_data:
        individual_lines_in_merged.extend(item['merged_line'])

    df = df_join.with_columns(
        pl.when(
            (pl.col('std_throughput') > 0) & 
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('actual_throughput') / pl.col('std_throughput')) * 100)
        .otherwise(None)
        .round(2)
        .alias('performance_rate')
    ).select(
        ['line', 'shift', 'performance_rate'] if not is_total else ['line', 'performance_rate']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    if is_fixed:
        df = df.with_columns(
            pl.col('performance_rate')
            .clip(upper_bound=100)
            .round(2)
            .alias('performance_rate')
        ).sort(['line', 'shift'] if not is_total else ['line'])

    return df.to_dicts() if is_json else df

def quality_rate(data: dict, output_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):

    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    df_output_fg = output_data.filter(
        pl.col('type').str.contains('Finished Good (kg)', literal=True)
        ).group_by(
            ['line', 'shift'] if not is_total else ['line']
        ).agg(pl.col('total_output').sum().alias('total_output_fg'))

    df_output_total = output_data.group_by(
            ['line', 'shift'] if not is_total else ['line']
        ).agg(pl.col('total_output').sum())

    df_join = (
            df.join(df_output_fg, on=['line', 'shift'] if not is_total else ['line'], how='left')
            .with_columns(pl.col('total_output_fg').fill_null(0))
        ).join(
            df_output_total, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(pl.col('total_output').fill_null(0))

    individual_lines_in_merged = []
    for item in merged_line_data:
        individual_lines_in_merged.extend(item['merged_line'])

    df = df_join.with_columns(
        pl.when(
            (pl.col('total_output') > 0) & 
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('total_output_fg') / pl.col('total_output')) * 100)
        .otherwise(None)
        .round(2)
        .alias('quality_rate')
    ).select(
        ['line', 'shift', 'quality_rate'] if not is_total else ['line', 'quality_rate']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    return df.to_dicts() if is_json else df

def overall_equipment_effectiveness(data: dict, availability_rate_data: pl.DataFrame, performance_rate_data: pl.DataFrame, quality_rate_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):
    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()
    
    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    df_join = df.join(
            availability_rate_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('availability_rate').fill_null(0)
        ).join(
            performance_rate_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('performance_rate').fill_null(0)
        ).join(
            quality_rate_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('quality_rate').fill_null(0)
        )

    individual_lines_in_merged = []
    for item in merged_line_data:
        individual_lines_in_merged.extend(item['merged_line'])

    df = df_join.with_columns(
        pl.when(
            (pl.col('availability_rate') * pl.col('performance_rate') * pl.col('quality_rate') != 0) &
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('availability_rate') * pl.col('performance_rate') * pl.col('quality_rate')) / 10000)
        .otherwise(None)
        .round(2)
        .alias('oee')
    ).select(
        ['line', 'shift', 'oee'] if not is_total else ['line', 'oee']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    return df.to_dicts() if is_json else df

def utilization(data: dict, total_time_data: pl.DataFrame, scheduled_downtime_data: pl.DataFrame, not_scheduled_downtime_data: pl.DataFrame, performance_loss_data: pl.DataFrame, defect_rework_loss_data: pl.DataFrame, value_operating_time_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):

    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()    

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    individual_lines_in_merged = []
    for item in merged_line_data:
        individual_lines_in_merged.extend(item['merged_line'])

    scheduled_downtime_data = scheduled_downtime_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1'] if not is_total else ['line', 'machine_losses_lvl_1']
    ).agg(
        pl.col('total_scheduled_downtime').sum().alias('value')
    )

    not_scheduled_downtime_data = not_scheduled_downtime_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1'] if not is_total else ['line', 'machine_losses_lvl_1']
    ).agg(
        pl.col('total_not_scheduled_downtime').sum().alias('value')
    )

    performance_loss_data = performance_loss_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1'] if not is_total else ['line', 'machine_losses_lvl_1']
    ).agg(
        pl.col('performance_loss' if not is_total else 'total_performance_loss').sum().alias('value')
    )

    defect_rework_loss_data = defect_rework_loss_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1'] if not is_total else ['line', 'machine_losses_lvl_1']
    ).agg(
        pl.col('defect_rework_loss' if not is_total else 'total_defect_rework_loss').sum().alias('value')
    )

    list_df = [
        scheduled_downtime_data,
        not_scheduled_downtime_data,
        performance_loss_data,
        defect_rework_loss_data
    ]

    df_losses = pl.concat(list_df).rename({
        'machine_losses_lvl_1': 'category'
    })

    df_losses = df_losses.join(
            total_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('total_time').fill_null(0)
        )

    df_losses = df_losses.with_columns(
        pl.when(
            (pl.col('total_time') > 0) &
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('value') / pl.col('total_time')) * 100)
        .otherwise(None)
        .round(2)
        .alias('utilization')
    ).select(
        ['line', 'shift', 'category', 'utilization'] if not is_total else ['line', 'category', 'utilization']
    ).sort(['line', 'shift'] if not is_total else ['line'])


    df = df.with_columns(
        pl.lit('Utilization').alias('category')
    )

    df_join = df.join(
            total_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('total_time').fill_null(0)
        ).join(
            value_operating_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('value_net_operating_time').fill_null(0)
        )

    df = df_join.with_columns(
        pl.when(
            (pl.col('total_time') > 0) &
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('value_net_operating_time') / pl.col ('total_time')) * 100)
        .otherwise(None)
        .round(2)
        .alias('utilization')
    ).select(
        ['line', 'shift', 'category', 'utilization'] if not is_total else ['line', 'category', 'utilization']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    df = pl.concat(
        [df, df_losses]
        ).sort(
            ['line', 'shift', 'category'] if not is_total else ['line', 'category']
        )

    return df.to_dicts() if is_json else df

def utilization_details(data: dict, total_time_data: pl.DataFrame, scheduled_downtime_data: pl.DataFrame, not_scheduled_downtime_data: pl.DataFrame, performance_loss_data: pl.DataFrame, defect_rework_loss_data: pl.DataFrame, is_total: bool = False, is_json: bool = False):

    required_keys = [
        'master_line', 
        'merged_line', 
        'master_shift', 
    ]

    if not all(key in data and data[key] for key in required_keys):
        return [] if is_json else pl.DataFrame()

    master_line_data = data['master_line']
    merged_line_data = data['merged_line']
    shift_data = data['master_shift']

    df_master_line = pl.from_dicts(master_line_data)
    merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
    df_merged = pl.DataFrame(merged_rows)

    df_all_lines = pl.concat([df_master_line, df_merged])

    df_shifts = pl.from_dicts(shift_data)

    df = df_all_lines
    if not is_total:
        df = (
            df_all_lines
            .join(df_shifts, how="cross")
        )

    individual_lines_in_merged = []
    for item in merged_line_data:
        individual_lines_in_merged.extend(item['merged_line'])

    scheduled_downtime_data = scheduled_downtime_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
    ).agg(
        pl.col('total_scheduled_downtime').sum().alias('value')
    )

    not_scheduled_downtime_data = not_scheduled_downtime_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
    ).agg(
        pl.col('total_not_scheduled_downtime').sum().alias('value')
    )

    performance_loss_data = performance_loss_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
    ).agg(
        pl.col('performance_loss' if not is_total else  'total_performance_loss').sum().alias('value')
    )

    defect_rework_loss_data = defect_rework_loss_data.group_by(
        ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
    ).agg(
        pl.col('defect_rework_loss' if not is_total else 'total_defect_rework_loss').sum().alias('value')
    )

    list_df = [
        scheduled_downtime_data,
        not_scheduled_downtime_data,
        performance_loss_data,
        defect_rework_loss_data
    ]

    df_losses = pl.concat(list_df)

    df_losses = df_losses.join(
            total_time_data, on=['line', 'shift'] if not is_total else ['line'], how='left'
        ).with_columns(
            pl.col('total_time').fill_null(0)
        )

    df = df_losses.with_columns(
        pl.when(
            (pl.col('total_time') > 0) &
            (~pl.col('line').is_in(individual_lines_in_merged))
        )
        .then((pl.col('value') / pl.col('total_time')) * 100)
        .otherwise(None)
        .round(2)
        .alias('utilization')
    ).select(
        ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3', 'utilization'] if not is_total else ['line',  'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3', 'utilization']
    ).sort(['line', 'shift'] if not is_total else ['line'])

    return df.to_dicts if is_json else df

In [4]:
data = await get_data((2, 3))

In [32]:
total_time_data_shift = total_time(data, (2, 3))
total_time_data = total_time(data, (2, 3), is_total=True)

sd_data_shift = scheduled_downtime(data)
sd_data_total = scheduled_downtime(data, is_total=True)

lt_data_shift = loading_time(total_time_data_shift, sd_data_shift)
lt_data_total = loading_time(total_time_data, sd_data_total, is_total=True)

nsd_data_shift = not_scheduled_downtime(data)
nsd_data_total = not_scheduled_downtime(data, is_total=True)

ot_data_shift = operating_time(lt_data_shift, nsd_data_shift)
ot_data_total = operating_time(lt_data_total, nsd_data_total, is_total=True)

output_data_shift = output(data)
output_data_total = output(data, is_total=True)

std_throughput_shift = standard_throughput(data)
std_throughput_total = standard_throughput(data, is_total=True)

act_throughput_shift = actual_throughput(ot_data_shift, output_data_shift)
act_throughput_total = actual_throughput(ot_data_total, output_data_total, is_total=True)

pl_data_shift = performance_loss(data, ot_data_shift, output_data_shift, std_throughput_shift)
pl_data_total = performance_loss_total(pl_data_shift)

not_data_shift = net_operating_time(ot_data_shift, pl_data_shift)
not_data_total = net_operating_time(ot_data_total, pl_data_total, is_total=True)

drl_data_shift = defect_rework_loss(data, output_data_shift, act_throughput_shift)
drl_data_total = defect_rework_loss_total(drl_data_shift)

vot_data_shift = value_operating_time(not_data_shift, drl_data_shift)
vot_data_total = value_operating_time(not_data_total, drl_data_total, is_total=True)

kode_pakan_gabungan_data = kode_pakan_gabungan(data)

availability_rate_shift = availability_rate(data, ot_data_shift, lt_data_shift)
availability_rate_total = availability_rate(data, ot_data_total, lt_data_total, is_total=True)

performance_rate_shift = performance_rate(data, act_throughput_shift, std_throughput_shift)
performance_rate_total = performance_rate(data, act_throughput_total, std_throughput_total, is_total=True)

performance_rate_shift_fixed = performance_rate(data, act_throughput_shift, std_throughput_shift, is_fixed=True)
performance_rate_total_fixed = performance_rate(data, act_throughput_total, std_throughput_total, is_total=True, is_fixed=True)

quality_rate_shift = quality_rate(data, output_data_shift)
quality_rate_total = quality_rate(data, output_data_total, is_total=True)

oee_data_shift = overall_equipment_effectiveness(data, availability_rate_shift, performance_rate_shift, quality_rate_shift)
oee_data_shift_fixed = overall_equipment_effectiveness(data, availability_rate_shift, performance_rate_shift_fixed, quality_rate_shift)
oee_data_total = overall_equipment_effectiveness(data, availability_rate_total, performance_rate_total, quality_rate_total, is_total=True)
oee_data_total_fixed = overall_equipment_effectiveness(data, availability_rate_total, performance_rate_total_fixed, quality_rate_total, is_total=True)

utilization_data_shift = utilization(data, total_time_data_shift, sd_data_shift, nsd_data_shift, pl_data_shift, drl_data_shift, vot_data_shift)
utilization_data_total = utilization(data, total_time_data, sd_data_total, nsd_data_total, pl_data_total, drl_data_total, vot_data_total, is_total=True)

utilization_details_data_shift = utilization_details(data, total_time_data_shift, sd_data_shift, nsd_data_shift, pl_data_shift, drl_data_shift)
utilization_details_data_total = utilization_details(data, total_time_data, sd_data_total, nsd_data_total, pl_data_total, drl_data_total, is_total=True)

In [36]:
utilization_details_data_total.filter(~pl.col('line').str.contains('Line 3'))['utilization'].sum()

382.63

In [ ]:
master_line_data = data['master_line']
merged_line_data = data['merged_line']
shift_data = data['master_shift']

df_master_line = pl.from_dicts(master_line_data)
merged_rows = [{"line": " & ".join(item['merged_line'])} for item in merged_line_data]
df_merged = pl.DataFrame(merged_rows)

df_all_lines = pl.concat([df_master_line, df_merged])

df_shifts = pl.from_dicts(shift_data)

df = df_all_lines
if not is_total:
    df = (
        df_all_lines
        .join(df_shifts, how="cross")
    )

individual_lines_in_merged = []
for item in merged_line_data:
    individual_lines_in_merged.extend(item['merged_line'])

scheduled_downtime_data = sd_data_shift.group_by(
    ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
).agg(
    pl.col('total_scheduled_downtime').sum().alias('value')
)

not_scheduled_downtime_data = nsd_data_shift.group_by(
    ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
).agg(
    pl.col('total_not_scheduled_downtime').sum().alias('value')
)

performance_loss_data = pl_data_shift.group_by(
    ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
).agg(
    pl.col('performance_loss' if not is_total else 'total_performance_loss').sum().alias('value')
)

defect_rework_loss_data = drl_data_shift.group_by(
    ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3']
).agg(
    pl.col('defect_rework_loss' if not is_total else 'total_defect_rework_loss').sum().alias('value')
)

list_df = [
    scheduled_downtime_data,
    not_scheduled_downtime_data,
    performance_loss_data,
    defect_rework_loss_data
]

df_losses = pl.concat(list_df)

df_losses = df_losses.join(
        total_time_data_shift, on=['line', 'shift'] if not is_total else ['line'], how='left'
    ).with_columns(
        pl.col('total_time').fill_null(0)
    )

df = df_losses.with_columns(
    pl.when(
        (pl.col('total_time') > 0) &
        (~pl.col('line').is_in(individual_lines_in_merged))
    )
    .then((pl.col('value') / pl.col('total_time')) * 100)
    .otherwise(None)
    .round(2)
    .alias('utilization')
).select(
    ['line', 'shift', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3', 'utilization'] if not is_total else ['line', 'machine_losses_lvl_1', 'machine_losses_lvl_2', 'machine_losses_lvl_3', 'utilization']
).sort(['line', 'shift'] if not is_total else ['line'])

df

1147.8799999999999